# 早期OOF差区间及历史控制器资格
复用0/2/5bp全部原生账户，精确现金/SOXL/费用对数归因，报告全部年份、季度与固定前置信息状态。独立核对旧控制器冻结包，不重新训练或拼接未来模型。

In [1]:
from pathlib import Path
from io import BytesIO
import os,json,hashlib
import numpy as np,pandas as pd,qlib
from qlib.workflow import R
from src.workflow.minute_action_value import _source
from src.analysis.decision_accounting import attribute_account
from src.data.nas_market_data import archive_nas_recorder
root=Path('/Users/rziiiii/dev/workspace');card=root/'research/tangent_oof_failure_20261015';p=json.loads((card/'plan.json').read_text());provider=root/'data/qlib/full_path_tiered_allocation_20261013';os.environ['MLFLOW_ALLOW_FILE_STORE']='true';qlib.init(provider_uri=str(provider),region='us')
source,audit=_source(root,p['source'],['inputs/frame',*[f'analysis/native/{family}_{cost}' for family in ['reference','soxx'] for cost in p['slippage_bps']]]);training,ta=_source(root,p['training_source'],['inputs/daily_frame'])
frame=source['inputs/frame'];observed=training['inputs/daily_frame'].reindex(pd.DatetimeIndex(frame.signal_date));assert observed[['soxx_ret5','soxx_ret20']].notna().all().all()
a=observed.soxx_ret5.ge(0).to_numpy();b=observed.soxx_ret20.ge(0).to_numpy();states=np.select([a&b,a&~b,~a&b],['uptrend','rebound','pullback'],default='decline')
terms=['reference_cash','additional_cash','soxl_relative','recovery_gross','execution_cost','benchmark_residual','native_reconciliation_residual','relative_log'];tables={group:[] for group in ['year','quarter','regime']};daily={}
for cost in p['slippage_bps']:
 d=attribute_account(frame,source[f'analysis/native/reference_{cost}'],source[f'analysis/native/soxx_{cost}'],initial_cash=p['initial_cash'],cash_reference=p['cash_reference']);d['year']=d.trade_date.dt.year;d['quarter']=d.trade_date.dt.to_period('Q').astype(str);d['regime']=states;daily[str(cost)]=d
 for group in tables:
  result=d.groupby(group)[terms].sum();result['sessions']=d.groupby(group).size();result['mean_cash_weight_after_open']=d.groupby(group).cash_weight_after_open.mean();result['slippage_bps']=cost;np.testing.assert_allclose(result[terms].sum(),d[terms].sum(),atol=1e-12);tables[group].append(result.reset_index())
tables={key:pd.concat(value,ignore_index=True) for key,value in tables.items()}
legacy=root/p['legacy_card'];manifest_path=legacy/'frozen_bundle_manifest.json';assert hashlib.sha256(manifest_path.read_bytes()).hexdigest()==p['legacy_manifest_sha256'];manifest=json.loads(manifest_path.read_text());s=json.loads((legacy/'summary.json').read_text());assert s['recorder_id']==manifest['recorder_id'];names=['open_action_value_models.pkl','training_diagnostics.csv','dataset_audit.json','runtime_models.json'];payloads={};identities={}
with R.uri_context(s['recorder_uri']):
 old=R.get_recorder(recorder_id=s['recorder_id'],experiment_name=legacy.name)
 for name in names:
  entry=next(e for e in manifest['artifacts'] if e['path'].endswith('/'+name));relative=entry['path'].split('/artifacts/',1)[1];path=Path(old.download_artifact(relative));data=path.read_bytes();assert hashlib.sha256(data).hexdigest()==entry['sha256'];payloads[name]=data;identities[name]=entry['sha256']
 models=old.load_object(legacy.name+'/open_action_value_models.pkl')
diag=pd.read_csv(BytesIO(payloads['training_diagnostics.csv']));runtime=json.loads(payloads['runtime_models.json']);evidence={'legacy_recorder_id':s['recorder_id'],'artifact_hashes':identities,'saved_annual_open_years':sorted(int(y) for y in models),'saved_runtime_models':list(runtime),'annual_recovery_artifact_present':any('recovery' in e['path'] and 'models' in Path(e['path']).name and 'frozen' not in Path(e['path']).name for e in manifest['artifacts']),'source_scope':'this pinned frozen bundle only; not a repository-wide absence proof'}
policy={'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/*','analysis/daily']}
with R.start(experiment_name=card.name,uri=str(card/'qlib_recorder')):
 rec=R.get_recorder();rec.save_objects(artifact_path='inputs',plan=p,source_audit=audit,training_source_audit=ta,legacy_artifacts=payloads,legacy_bundle_manifest=manifest);rec.save_objects(artifact_path='analysis',daily=daily,tables=tables,controller_evidence=evidence);rec.log_metrics(reused_accounts=6,registered_artifact_count=8)
summary={'experiment':card.name,'decision':'diagnostic_only_no_promotion','new_fits':0,'new_backtests':0,'provider_uri':str(provider),'region':'us','as_of':'2026-10-09','recorder_id':rec.id,'experiment_id':str(rec.experiment_id),'recorder_uri':str(card/'qlib_recorder'),'recorder_local_dir':str(card/'qlib_recorder'/str(rec.experiment_id)/rec.id),'artifact_policy':policy,'scope':p['scope'],'objective_achieved':False}
for name,value in tables.items():value.to_csv(card/(name+'_attribution.csv'),index=False)
diag.to_csv(card/'legacy_open_training_audit.csv',index=False);(card/'controller_evidence.json').write_text(json.dumps(evidence,indent=2))
import yaml
(card/'workflow.yaml').write_text(yaml.safe_dump({'experiment':card.name,'provider_uri':str(provider),'artifact_policy':policy}));binding=json.loads((root/p['source']['card']/'nas_binding.json').read_text());binding.pop('evidence',None);(card/'nas_binding.json').write_text(json.dumps(binding,indent=2));art=Path(summary['recorder_local_dir'])/'artifacts';(card/'verification_manifest.json').write_text(json.dumps({'recorder_id':rec.id,'entries':[{'path':str(f.relative_to(card)),'sha256':hashlib.sha256(f.read_bytes()).hexdigest()} for f in sorted(art.rglob('*')) if f.is_file()]},indent=2))
summary['nas_evidence']=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rec.id,experiment_id=str(rec.experiment_id));(card/'summary.json').write_text(json.dumps(summary,indent=2));print(tables['year'].query('slippage_bps == 2').to_string(index=False));print(evidence)


# 早期OOF损失归因与控制器证据

0拟合、0新增回测。复用2020–2022的756区间、0/2/5bp参考与SOXX共6个已核验Qlib账户，使用既有decision_accounting恒等式分解实际持仓的相对收益。逐日各项之和等于相对对数收益，累加后等于期末相对财富的对数；全部年份、季度和四个固定前置信息状态均保留，未挑最差区间调参。

2bp按交易年归因（单位为相对对数财富贡献，不是收益百分点）：

| 年 | 现金两部分合计 | SOXL相对项 | 执行费用 | 相对对数合计 | 平均开盘后现金 |
|---|---:|---:|---:|---:|---:|
| 2020 | -0.41986 | -0.03120 | -0.00443 | -0.45486 | 54.11% |
| 2021 | -0.14858 | +0.00024 | -0.00414 | -0.15228 | 31.68% |
| 2022 | +0.27102 | -0.06123 | -0.00162 | +0.20797 | 70.41% |

表中未单列的SOXX现金/整股残差和原生核对残差均保留在CSV恒等式中。现金分为最多15%的参考现金与其余现金，仅为固定会计拆分；其余现金不等于可直接移除的风控仓位。2020/2021主要落后项是现金机会成本，2022现金保护抵消部分SOXL相对损失。费用并非主要缺口，但不能由归因推断降低现金会满足30%回撤门。恢复贡献为0，因为此参考本就没有学习型恢复，不能据此证明恢复控制器没有价值。

旧控制器来源审计：已核对stage1_open_action_value_net_fees_20260928冻结bundle及4个指定Recorder对象哈希。open_action_value_models.pkl包含2017–2026年度键；runtime仅保存frozen_open_action_value_model和frozen_recovery_model。该冻结包没有年度恢复模型产物，不推断全仓库不存在任何其他恢复证据。

年度开盘训练资格不能简化为1月1日：旧2020模型的最后标签成熟2020-01-02 09:31 ET，原首个测试决策2020-01-03 09:31；2021对应1月4日成熟、1月5日首决策；2022对应1月3日成熟、1月4日首决策。本次OOF参考各年从更早首交易日开始，不能仅按trade_year就认定旧模型已覆盖首日。需区分：这些末标签在首日09:31成熟，早于新决策09:46，因此本项并未证明首日存在未来标签泄漏；缺的是按新决策cutoff核对训练行集与模型身份的证据。原测试起点较晚也不等于模型最早可能使用时点。2023冻结恢复模型不能回填2020。

源代码说明旧开发期恢复模型曾按年拟合，但当前包没有其模型快照；冻结恢复模型训练至2022。旧开发配置输出与外层冻结扩展在代码中分开，因此不能仅看source_model_frozen标记就断言全部旧开发预测都用了未来模型，也不能仅看年度键便断言完整时序已合格。

决策diagnostic_only_no_promotion。下一项需要在当时成熟标签和合格OOF配置基础上重建年度恢复训练证据，并固定预测可用时点；未完成前不拼接完整早期控制器。重点是防守后恢复信息，而非继续费用微调；不依据本次季度/状态结果调等待长度、阈值或翻转信号。该参考不是完整生产策略，模型结构及研究规则已受历史选择影响，仍无独立稳定净SOXX超额证据。

生产未改，目标未完成。源模型审计、逐日归因及汇总均在Qlib Recorder，NAS冷恢复逐对象哈希通过。
